# TIAToolbox Hands‑On: Mini End‑to‑End Computational Pathology

In this session, you will explore the fundamentals of handling whole-slide images (WSIs), extracting patches, performing stain normalisation, and using deep learning models (ie, HoVer-Net) for tissue and nuclei analysis using the [TIAToolbox](https://github.com/TissueImageAnalytics/tiatoolbox).

Keep a tab open for [the documentation](https://tia-toolbox.readthedocs.io/en/latest/index.html), such that you can easily access it to help you solve each question.

Before starting, remember to already save this notebook in your own account to be able to save your progress.

And don't forget to select the runtime type to T4 GPU.


## Environment setup


In [ ]:
# Install some system-level required packages
!apt-get -y install libopenjp2-7-dev libopenjp2-tools

In [ ]:
# Install TIA Toolbox and OpenSlide (it might take a few minutes)
# Need to specify a specific version of numcodecs because of dependency issue
!pip install tiatoolbox openslide-bin numcodecs==0.12.1

Execute the cell below just with imports and utility functions which we will use later.

In [ ]:
import os
from pathlib import Path
from pprint import pprint

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from tiatoolbox.utils.misc import download_data, select_device
from tiatoolbox.utils import imread, imwrite
from tiatoolbox.wsicore.wsireader import WSIReader
from tiatoolbox.tools import patchextraction
from tiatoolbox.tools.stainnorm import MacenkoNormalizer
from tiatoolbox.data import small_svs, stain_norm_target

from tiatoolbox.utils.visualization import overlay_prediction_mask, overlay_prediction_contours

# Reproducibility (within reason)
rng = np.random.default_rng(0)

mpl.rcParams["figure.dpi"] = 150
mpl.rcParams["figure.facecolor"] = "white"
plt.rcParams.update({"font.size": 9})

def show_side_by_side(img1, img2, titles=("A", "B"), figsize=(9, 4)):
    fig, axs = plt.subplots(1, 2, figsize=figsize)
    axs[0].imshow(img1); axs[0].set_title(titles[0]); axs[0].axis("off")
    axs[1].imshow(img2); axs[1].set_title(titles[1]); axs[1].axis("off")
    plt.show()

def show_grid(patches, n=16, title="Patches", figsize=(8, 8)):
    n = min(n, len(patches))
    k = int(np.ceil(np.sqrt(n)))
    fig, axs = plt.subplots(k, k, figsize=figsize)
    fig.suptitle(title)
    axs = np.array(axs).reshape(-1)
    for i in range(k*k):
        axs[i].axis("off")
        if i < n:
            axs[i].imshow(patches[i])
    plt.show()

device = 'cuda' # Just use 'cpu' if you are not using a runtime with GPU

You will now download a region of a WSI (rather than an entire WSI) to ensure you don't hit Colab's limits.

If you want to explore this notebook even faster, notice that commented lines with code to get an even smaller WSI.

In [ ]:
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

# Small WSI
# wsi_small_path = Path(small_svs())
# print("Small WSI:", wsi_small_path, "size(MB)=", wsi_small_path.stat().st_size / 1e6)

# "Moderate" WSI (Colab-friendly)
wsi_med_path = Path(download_data(
    "https://tiatoolbox.dcs.warwick.ac.uk/sample_wsis/wsi4_12k_12k.svs",
    data_dir / "wsi4_12k_12k.svs",
))
print("Moderate WSI:", wsi_med_path, "size(MB)=", wsi_med_path.stat().st_size / 1e6)


## WSI loading and visualisation


We can open a WSI image using `WSIReader.open()`. Have a closer look to the information in the `info_dict` object below - does it make sense?

In [ ]:
wsi = WSIReader.open(wsi_med_path)

info_dict = wsi.info.as_dict()
pprint(info_dict)

In [ ]:
# Use `slide_thumbnail()` to visualise the WSI image. Notice the different ways
#  to read the image

thumb = wsi.slide_thumbnail(resolution=1.25, units="power")
#thumb = wsi.slide_thumbnail(resolution=0.5, units="mpp") # 0.5 microns per-pixel
#thumb = wsi.slide_thumbnail(resolution=0, units="level") # level 0 at the pyramid layer (highest resolution)
plt.figure(figsize=(6,5))
plt.imshow(thumb)
plt.title("WSI thumbnail @1.25x")
plt.axis("off")
plt.show()

In [ ]:
# Sanity checking the shape of the returned image
# Check this shape every time you run `slide_thumbnail()` with different parameters.
thumb.shape

### Exercise

Using `mpp` and `slide_dimensions`, estimate the physical slide size in **mm** (width × height).

**Tip:** `mpp` is µm per pixel, and 1000 µm = 1 mm.

In [ ]:
# TODO: Compute the physical size in mm (w, h) here


### Exercise
Use the function `read_rect()` to select a region of the WSI at 20x and 5x magnification.

The starting coordinates and size are already provided, and you can use the utility function `show_side_by_side()` to see both ROIs. ROI stands for Region of Interest.



In [ ]:
roi_xy = (2000, 2500)      # top-left (x, y)
roi_size = (1024, 1024)    # (width, height) output size

# TODO: Do the rest of the exercise here


### Exercise

Tweak coordinates to find two different ROIs: one with more nuclei, and another with more connectivity tissue. Do you remember how they should look like?


In [ ]:
# TODO: define roi_xy_A, roi_xy_B and visualise at 20x vs 5x
roi_xy_A = (, )
roi_xy_B = (, )


## Patch extraction

TIAToolbox offers `tissue_mask()` for quick detection/masking of regions that are not "glass"/background, such that you don't select patches that only contain background.

Below here you can see it practice.



In [ ]:
mask_reader = wsi.tissue_mask(resolution=1.25, units="power")
mask_thumb = mask_reader.slide_thumbnail(resolution=1.25, units="power")

show_side_by_side(thumb, mask_thumb, titles=("WSI thumbnail", "Tissue mask thumbnail"), figsize=(10,4))

### Exercise

Compute the fraction of pixels flagged as tissue in the mask of the `thumb` object.


In [ ]:
# TODO: compute tissue fraction here

### Exercise

Using the function [`patchextraction.get_patch_extractor()`](https://tia-toolbox.readthedocs.io/en/latest/_autosummary/tiatoolbox.tools.patchextraction.get_patch_extractor.html#tiatoolbox.tools.patchextraction.get_patch_extractor), extract patches of size (224, 224). Notice the ** kwargs** parameter that will be passed to `PatchExtractor`.

You can use the utility function `show_grid()` to check the patches extracted.


In [ ]:
patch_size = (224, 224)
patch_res = 0.5
patch_units = "mpp"
grid_patches = [] # Where you'll save the extracted patches to use later

# TODO: Complete exercise here


## Stain normalisation

In the code below you will get a stained image to serve as our `target` image to run the Macenko algorithm on.

In [ ]:
target_img = stain_norm_target()
plt.figure(figsize=(4,4))
plt.imshow(target_img)
plt.title("Stain normalisation target (provided by TIAToolbox)")
plt.axis("off")
plt.show()

norm = MacenkoNormalizer()
norm.fit(target_img)
print("Macenko normaliser fitted.")

### Exercise

Transform the patches you previously extracted to `grid_patches` using the Macenko normalisation, and plot each one before and after. You can use the utility function `show_side_by_side()` to help.

**Tip:** Normalise and visualise only 4 patches for now, such that you can go quicker to the following exercises.

In [ ]:
norm_patches = [] # Where you'll save the normalised images

# TODO: normalise the images in `grid_patches`


### Exercise

Compute per‑channel statistics (mean/std in RGB) before vs after normalisation for 10 patches in `grid_patches`.

In [ ]:
# TODO: compute RGB mean/std before vs after for a batch of patches here


## Nuclei instance segmentation using HoVer‑Net

We’ll run TIAToolbox’s pretrained HoVer‑Net via `NucleusInstanceSegmentor` on a single ROI which you can extract from the `wsi` object, not the full slide.

**Runtime tip:** If you are using CPU runtime, keep ROI modest by changing the size of the ROI and resolution.

**Suggestion:** Use `pretrained_model="hovernet_fast-pannuke"`

In [ ]:
from tiatoolbox.models.engine.nucleus_instance_segmentor import NucleusInstanceSegmentor
import joblib

roi_for_hover = wsi.read_rect(roi_xy, (1024, 1024), resolution=20, units="power")

# TODO: Run nuclei segmentation here


In [ ]:
# Assuming the output of your instance segmentor's predict() was saved in a
#  variable named `hover_out`, you can use the following code to visualise the
#  predicted output.
tile_preds = joblib.load(f"{hover_out[0][1]}.dat")

color_dict = {
    0: ("background", (255, 165, 0)),
    1: ("neoplastic epithelial", (255, 0, 0)),
    2: ("Inflammatory", (255, 255, 0)),
    3: ("Connective", (0, 255, 0)),
    4: ("Dead", (0, 0, 0)),
    5: ("non-neoplastic epithelial", (0, 0, 255)),
}

overlay = overlay_prediction_contours(
    canvas=roi_for_hover,
    inst_dict=tile_preds,
    draw_dot=False,
    type_colours=color_dict,
    line_thickness=2,
)

show_side_by_side(roi_for_hover, overlay, titles=("ROI", "HoVer-Net overlay"), figsize=(12,4))
print("Detected nuclei:", len(tile_preds))

### Exercise

Choose a different ROI and rerun HoVer‑Net. Also play a bit with the `line_thickness` and `draw_dot` parameters in  `overlay_prediction_contours()`.

**Suggestion:** Use the previous ROI you found that has more connective tissue, to see the differences.

In [ ]:
# TODO: rerun on a new ROI and summarise nucleus type counts here

## Morphological features

The following code can be used to compute per-nucleus morphology information, specifically area, eccentricity, and equivalent diameter.

It already calculates this information for the previous Hover-Net predictions that were stored in the `tile_preds` variable.


In [ ]:
from skimage.draw import polygon
from skimage.measure import regionprops

def inst_dict_to_features(inst_dict, canvas_shape):
    feats = []
    for nuc_id, nuc in inst_dict.items():
        contour = np.asarray(nuc["contour"], dtype=np.int32)  # (N,2) in x,y
        if contour.ndim != 2 or contour.shape[0] < 3:
            continue

        x, y, w_, h_ = nuc["box"]
        x0, y0 = max(0, x), max(0, y)
        x1, y1 = min(canvas_shape[1], x + w_), min(canvas_shape[0], y + h_)
        if x1 <= x0 or y1 <= y0:
            continue

        cx = contour[:, 0] - x0
        cy = contour[:, 1] - y0
        rr, cc = polygon(cy, cx, shape=(y1 - y0, x1 - x0))
        mask = np.zeros((y1 - y0, x1 - x0), dtype=np.uint8)
        mask[rr, cc] = 1

        props = regionprops(mask)
        if not props:
            continue
        p = props[0]

        feats.append({
            "nuc_id": nuc_id,
            "type": nuc["type"],
            "area_px2": float(p.area),
            "eccentricity": float(p.eccentricity),
            "equiv_diameter": float(p.equivalent_diameter),
            "centroid_x": float(nuc["centroid"][0]),
            "centroid_y": float(nuc["centroid"][1]),
        })
    return pd.DataFrame(feats)

df_feat = inst_dict_to_features(tile_preds, roi_for_hover.shape[:2])
display(df_feat.head())
print("Nuclei with features:", len(df_feat), "/", len(tile_preds))

### Exercise

Run HoVer-Net on stain-normalised ROI (use the `roi_for_hover` variable you defined before), and use the `inst_dict_to_features()` function to compare the morphological features before and after normalisation.

In [ ]:
# TODO: Run HoVer-Net on stain-normalised ROI and check the morphological features here
# TODO: Plot the distributions to more easily visualise the differences


### Exercise

Using Pandas' `groupby()` function, summarise the features variability by nucleus type (notice `inst_dict_to_features()` has a "type" column). Do you see any significant difference? If you have time, try the Vahadane normalisation to see if it is any different.

You don't necessarily need to use `groupby()` to calculate the mean and standard deviation of area, eccentricity, diameter for each type, but this will make your code much cleaner.


**Tip:** You will have to use the `df_feat` variable, as well as the equivalent one for the normalised ROI.

In [ ]:
# TODO: groupby summaries before and after normalisation here.
